In [ ]:
# Phase 4/7 (manual source): CBP Airport Wait Times, DFW Terminal D.
# Manually exported from awt.cbp.gov (JS-rendered dashboard, no automatable
# export endpoint -- see docs/decisions/0001) and already clean: real CSV
# header, no merged cells, no PDF wrangling. Lands as-is into lh_bronze.

from pyspark.sql.functions import col, current_timestamp, lit, to_date
from pyspark.sql.types import IntegerType

WORKSPACE_ID = "16b2a9c6-6418-485f-865f-c47442b8b6c6"
LH_BRONZE_ID = "09a8b1f5-f33c-4eed-912d-40b9cbdc9b06"
ABFSS_ROOT = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{LH_BRONZE_ID}"

csv_path = f"{ABFSS_ROOT}/Files/cbp/awt_dfw_terminal_d.csv"
df = spark.read.option("header", True).csv(csv_path)
print(f"read {df.count():,} rows from {csv_path}")


In [ ]:
int_cols = [
    "Average_0_15_PassengerCount", "Average_121_PassengerCount", "Average_16_30_PassengerCount",
    "Average_31_45_PassengerCount", "Average_46_60_PassengerCount", "Average_61_90_PassengerCount",
    "Average_91_120_PassengerCount", "AverageWait", "BoothsUsed", "ExcludedPassengerCount",
    "FlightCount", "GlobalEntryPassengerCount", "MaxWait", "NonUsaPassengerAverageWait",
    "NonUsaPassengerCount", "NonUsaPassengerMaxWait", "TotalPassengerCount",
    "UsaPassengerAverageWait", "UsaPassengerCount", "UsaPassengerMaxWait",
]
for c in int_cols:
    df = df.withColumn(c, col(c).cast(IntegerType()))

df = df.withColumn("FlightDate", to_date(col("FlightDate")))
df = (
    df.withColumn("_source_file", lit("awt_dfw_terminal_d.csv"))
    .withColumn("_loaded_at", current_timestamp())
)
df.printSchema()


In [ ]:
table_path = f"{ABFSS_ROOT}/Tables/cbp_wait_hour"
df.write.format("delta").mode("overwrite").save(table_path)
print(f"wrote {table_path}")


In [ ]:
check = spark.read.format("delta").load(table_path)
print(f"cbp_wait_hour row count: {check.count():,}")
check.select("FlightDate", "HourRange", "TotalPassengerCount", "AverageWait", "MaxWait").show(5)
